<a href="https://colab.research.google.com/github/avi-dot-ai/W1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

Lane: Ranking Signal Analysis. This notebook frames a decision-support scoring problem, not a claim about Google's ranking algorithm or a causal experiment.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: scoring (regression). For a content team deciding which pages deserve ranking investigation first, the output will be a page-level estimated search-position score. Lower average position is better, so pages with worse estimated positions can be placed earlier in an optimisation queue. This is scoring rather than query-result ranking because the starter data has one row per page and no query–document comparison groups.
Decision and action: An SEO specialist reviews the highest-priority pages, diagnoses their search intent/content/freshness issues, and chooses which pages to refresh or test. A wrong score can waste limited editorial or engineering time on pages unlikely to benefit, or leave a weak-performing page unreviewed.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

import pandas as pd, numpy as np
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists())
data_path = repo_root / 'data/raw/content_refresh_anonymized.csv'
pages = pd.read_csv(data_path)
print(f'Loaded {len(pages):,} content items from {pages.client_id.nunique()} pseudonymized clients.')

Loaded 30,000 content items from 32 pseudonymized clients.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target proxy is avg_position, the observed mean Google Search Console position for a page during this snapshot's trailing 90-day window. It is an observed measurement, not a label constructed from a rule. Lower values indicate better positions. The 1,205 rows where avg_position == 0 are not rank zero; the data dictionary says they have no position data, so they are excluded from the target.
This proxy supports prioritisation, not a claim that a feature causes a ranking change. A later warehouse task can replace it with a strictly future position outcome.

In [3]:
scoring_pages = pages.loc[pages['avg_position'].gt(0)].copy()
scoring_pages['position_target'] = scoring_pages['avg_position']

print(f'Usable observed position targets: {len(scoring_pages):,}')
print(f'Missing/no-position rows excluded: {(pages.avg_position == 0).sum():,}')
scoring_pages[['content_id', 'client_id', 'avg_position', 'position_target']].head()

Usable observed position targets: 28,795
Missing/no-position rows excluded: 1,205


,content_id,client_id,avg_position,position_target
0,content_304f48230142,client_f369cb89fc,10.6,10.6
1,content_a1fb4e703a9e,client_4e07408562,20.3,20.3
2,content_9aa793d4d895,client_7f2253d7e2,36.5,36.5
3,content_331d6c4de07b,client_19581e27de,6.2,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,44.0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: mean absolute error (MAE) in average-position points, evaluated with a client-holdout split. MAE is easy to interpret: it is the typical number of position points by which a page score is wrong. I will compare it with a median-position baseline and call a model useful only if it has lower holdout MAE. This measures predictive accuracy; it does not prove that changing a signal will improve rank.
The observed median valid position is shown below as the simple baseline prediction.

In [4]:
median_position_baseline = scoring_pages['position_target'].median()
baseline_mae = (scoring_pages['position_target'] - median_position_baseline).abs().mean()
print(f'Median-position baseline: {median_position_baseline:.1f}')
print(f'In-sample reference MAE: {baseline_mae:.2f} position points')
print('Final comparison will use holdout clients, not this in-sample reference.')

Median-position baseline: 11.4
In-sample reference MAE: 10.38 position points
Final comparison will use holdout clients, not this in-sample reference.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content item (page) for one client, summarised over the trailing 90-day snapshot. content_id is unique, and client_id identifies the client for grouped evaluation; neither ID is a model feature. The displayed columns are candidate page/context signals and the observed position proxy.

In [5]:
display_columns = [
    'content_id', 'client_id', 'content_type', 'main_intent',
    'search_volume', 'competition', 'content_age_days',
    'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position'
]
print(f'One row per content item: {pages.content_id.nunique() == len(pages)}')
pages[display_columns].head()

One row per content item: True


,content_id,client_id,content_type,main_intent,search_volume,competition,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.0,0.67,187,20,3803,0.76,10.6
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,90.0,0.01,445,25,15320,0.05,20.3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.0,0.00,141,20,12581,0.09,36.5
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,10.0,0.00,463,22,11751,0.49,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.0,0.00,263,14,19140,0.13,44.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single rule such as “refresh the oldest page” or “work on the page with the most impressions” ignores interactions among search demand, competition, content type, intent, age, freshness, and engagement. A transparent scoring model can estimate their combined, non-linear pattern and be checked on unseen clients. Its output is a prioritised review queue for SEO/content specialists, not an automatic change to pages.
The model must still earn its place: if a simple baseline performs as well on client-holdout MAE, the team should use the simpler baseline. IDs, provider/model metadata, and leakage-prone trend fields will not be used as features.

In [6]:
candidate_features = [
    'search_volume', 'competition', 'content_type', 'main_intent',
    'word_count', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'ctr', 'engagement_rate', 'scroll_rate'
]
excluded_from_features = [
    'content_id', 'client_id', 'provider_used', 'model_used',
    'trend_direction', 'trend_pct', 'avg_position'
]
print('Candidate signals:', ', '.join(candidate_features))
print('Explicitly excluded:', ', '.join(excluded_from_features))

Candidate signals: search_volume, competition, content_type, main_intent, word_count, content_age_days, days_since_last_update, impressions_90d, ctr, engagement_rate, scroll_rate
Explicitly excluded: content_id, client_id, provider_used, model_used, trend_direction, trend_pct, avg_position


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

I named a scoring task and the decision it supports.

I defined an observed target proxy and handled the zero/no-data values.

I chose MAE and a median baseline before training.

I showed the real page-level dataframe and unit of analysis.

I explained why a model may help, and when a simple baseline should win.

I used careful, decision-support language and kept client IDs pseudonymous.